#  BanglaBERT — Multilabel Complaint Classification
### Thesis Project | Fixed & Complete Version
---
**Labels:** Billing, Refund, Delivery, Customer_Service, Product_Quality, Account, Technical, Network

##Step 1 — Install Libraries

In [ ]:
!pip install transformers scikit-learn openpyxl torch -q

##Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

print(' Libraries imported')
print(' Device:', 'GPU ' if torch.cuda.is_available() else 'CPU  (GPU recommend)')

✅ Libraries imported
🖥️ Device: GPU ✅


## Step 3 — Config

In [ ]:

TRAIN_FILE = '/content/Final_Cleaned_Labels (2).xlsx'
TEST_FILE  = '/content/test (1).xlsx'

MODEL_NAME  = 'csebuetnlp/banglabert'
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5
THRESHOLD   = 0.5
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


LABELS = ['Billing', 'Refund', 'Delivery', 'Customer_Service',
          'Product_Quality', 'Account', 'Technical', 'Network']

print('Config set')
print(f'   Labels ({len(LABELS)}):', LABELS)

✅ Config set
   Labels (8): ['Billing', 'Refund', 'Delivery', 'Customer_Service', 'Product_Quality', 'Account', 'Technical', 'Network']


##Step 4 — Upload Train & Test Files

In [ ]:
from google.colab import files
print(' Step 1: Final_Cleaned_Labels.xlsx upload  (train data)')
uploaded = files.upload()
print('\n Step 2: Processed_Data.xlsx upload  (test data)')
uploaded = files.upload()

📂 Step 1: Final_Cleaned_Labels.xlsx upload করো (train data)


Saving Final_Cleaned_Labels (2).xlsx to Final_Cleaned_Labels (2) (2).xlsx

📂 Step 2: Processed_Data.xlsx upload করো (test data)


Saving test (1).xlsx to test (1) (2).xlsx


##Step 5 — Load & Binarize Data

In [ ]:

mlb = MultiLabelBinarizer(classes=LABELS)

def load_and_binarize(file_path, fit=False):
    df = pd.read_excel(file_path).dropna(subset=['labels'])
    df['labels'] = df['labels'].str.strip()
    df['labels_list'] = df['labels'].apply(lambda x: [l.strip() for l in x.split(';') if l.strip()])

    y = mlb.fit_transform(df['labels_list']) if fit else mlb.transform(df['labels_list'])
    return df['text'].values, y

X_train, y_train = load_and_binarize(TRAIN_FILE, fit=True)
X_test,  y_test  = load_and_binarize(TEST_FILE,  fit=False)
print(f' Train: {len(X_train)} samples')
print(f' Test:  {len(X_test)} samples')
print(f' Labels: {mlb.classes_}')
print(f' y_train shape: {y_train.shape}')

✅ Train: 3278 samples
✅ Test:  795 samples
✅ Labels: ['Billing' 'Refund' 'Delivery' 'Customer_Service' 'Product_Quality'
 'Account' 'Technical' 'Network']
✅ y_train shape: (3278, 8)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['network'] will be ignored
  warnings.warn(


##Step 6 — Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ComplaintDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_loader = DataLoader(ComplaintDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(ComplaintDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

print('Tokenizer loaded')
print(f' Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

✅ Tokenizer loaded
✅ Train batches: 205 | Test batches: 50


##  Step 7 — Load BanglaBERT Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type='multi_label_classification'
).to(DEVICE)

print(' BanglaBERT model loaded')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'   Device: {DEVICE}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

✅ BanglaBERT model loaded
   Parameters: 110,623,496
   Device: cuda


## ✅ Step 8 — Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.BCEWithLogitsLoss()

print(' Training Started...')
print('=' * 45)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask)
        loss    = criterion(outputs.logits, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

print('=' * 45)
print(' Training শেষ!')

🚀 Training শুরু হচ্ছে...
Epoch 1/5  |  Loss: 0.4410
Epoch 2/5  |  Loss: 0.2732
Epoch 3/5  |  Loss: 0.1859
Epoch 4/5  |  Loss: 0.1353
Epoch 5/5  |  Loss: 0.1046
✅ Training শেষ!


##  Step 9 — Evaluation (Thesis Result)

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        logits = model(input_ids=ids, attention_mask=mask).logits
        preds  = (torch.sigmoid(logits) > THRESHOLD).cpu().numpy()
        all_preds.append(preds)
        all_true.append(batch['labels'].numpy())

all_preds = np.vstack(all_preds)
all_true  = np.vstack(all_true)
from sklearn.metrics import accuracy_score

print(f'Accuracy     : {accuracy_score(all_true, all_preds):.4f}')

print('=' * 55)
print('       BanglaBERT — Evaluation Results')
print('=' * 55)
print(classification_report(all_true, all_preds, target_names=LABELS, zero_division=0))
print(f'Hamming Loss : {hamming_loss(all_true, all_preds):.4f}')
print(f'F1 Micro     : {f1_score(all_true, all_preds, average="micro", zero_division=0):.4f}')
print(f'F1 Macro     : {f1_score(all_true, all_preds, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted  : {f1_score(all_true, all_preds, average="weighted", zero_division=0):.4f}')

Accuracy     : 0.6453
       BanglaBERT — Evaluation Results
                  precision    recall  f1-score   support

         Billing       0.72      0.78      0.75       119
          Refund       0.85      0.85      0.85        91
        Delivery       0.79      0.80      0.80       117
Customer_Service       0.67      0.87      0.76       126
 Product_Quality       0.80      0.76      0.78       100
         Account       0.78      0.88      0.83       161
       Technical       0.67      0.66      0.66       108
         Network       0.86      0.65      0.74        77

       micro avg       0.76      0.79      0.77       899
       macro avg       0.77      0.78      0.77       899
    weighted avg       0.76      0.79      0.77       899
     samples avg       0.77      0.81      0.77       899

Hamming Loss : 0.0656
F1 Micro     : 0.7735
F1 Macro     : 0.7703
F1 Weighted  : 0.7729


 STEP-10 Detailed Evaluation

In [ ]:

# Detailed Evaluation — Thesis & Research Paper
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             matthews_corrcoef, multilabel_confusion_matrix)

print('=' * 55)
print('        Detailed Evaluation Results')
print('=' * 55)

# ─── Overall Metrics ──────────────────────────────
print('\n Overall Metrics:')
print(f'  Accuracy (Exact Match) : {accuracy_score(all_true, all_preds):.4f}')
print(f'  F1 Micro               : {f1_score(all_true, all_preds, average="micro", zero_division=0):.4f}')
print(f'  F1 Macro               : {f1_score(all_true, all_preds, average="macro", zero_division=0):.4f}')
print(f'  F1 Weighted            : {f1_score(all_true, all_preds, average="weighted", zero_division=0):.4f}')
print(f'  Hamming Loss           : {hamming_loss(all_true, all_preds):.4f}')
print(f'  ROC-AUC (Macro)        : {roc_auc_score(all_true, all_preds, average="macro"):.4f}')

# ─── Per Label MCC ────────────────────────────────
print('\n Per-Label MCC (Matthews Correlation Coefficient):')
for i, label in enumerate(LABELS):
    mcc = matthews_corrcoef(all_true[:, i], all_preds[:, i])
    print(f'  {label:20s} : {mcc:.4f}')

# ─── Confusion Matrix ─────────────────────────────
print('\n Per-Label Confusion Matrix:')
print(f'  {"Label":20s}   TP    FP    FN    TN')
print('  ' + '-' * 45)
mcm = multilabel_confusion_matrix(all_true, all_preds)
for i, label in enumerate(LABELS):
    tn, fp, fn, tp = mcm[i].ravel()
    print(f'  {label:20s}  {tp:4d}  {fp:4d}  {fn:4d}  {tn:4d}')

        Detailed Evaluation Results

📊 Overall Metrics:
  Accuracy (Exact Match) : 0.6453
  F1 Micro               : 0.7735
  F1 Macro               : 0.7703
  F1 Weighted            : 0.7729
  Hamming Loss           : 0.0656
  ROC-AUC (Macro)        : 0.8691

📊 Per-Label MCC (Matthews Correlation Coefficient):
  Billing              : 0.7047
  Refund               : 0.8263
  Delivery             : 0.7612
  Customer_Service     : 0.7151
  Product_Quality      : 0.7490
  Account              : 0.7819
  Technical            : 0.6113
  Network              : 0.7258

📊 Per-Label Confusion Matrix:
  Label                  TP    FP    FN    TN
  ---------------------------------------------
  Billing                 93    36    26   640
  Refund                  77    14    14   690
  Delivery                94    25    23   653
  Customer_Service       110    54    16   615
  Product_Quality         76    19    24   676
  Account                141    39    20   595
  Technical             

##  Step 10 — Save Model

In [ ]:
model.save_pretrained('./banglabert_final')
tokenizer.save_pretrained('./banglabert_final')
print(' Model saved → ./banglabert_final/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved → ./banglabert_final/


##  Step 11 — Predict Function

In [ ]:
def predict(text, threshold=0.3):
    model.eval()
    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=mask).logits
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    results = [(LABELS[i], round(float(probs[i]), 3)) for i in range(len(LABELS)) if probs[i] > threshold]

    if not results:
        best = int(np.argmax(probs))
        results = [(LABELS[best], round(float(probs[best]), 3))]

    return results

test_texts = [
    'আমার অর্ডার ডেলিভারি হয়নি এবং টাকাও ফেরত পাইনি',
    'কাস্টমার সার্ভিস একদম বাজে, কেউ ফোন ধরে না',
    'নেটওয়ার্ক সমস্যার কারণে পেমেন্ট হয়নি',
    'লগইন করতে পারতেছি না অ্যাকাউন্টে ঢুকতে পারছি না',
]

print('=' * 50)
print('  Sample Predictions')
print('=' * 50)
for t in test_texts:
    print(f'\n📝 {t}')
    print(f'🏷️  {predict(t)}')

  Sample Predictions

📝 আমার অর্ডার ডেলিভারি হয়নি এবং টাকাও ফেরত পাইনি
🏷️  [('Refund', 0.976), ('Delivery', 0.329)]

📝 কাস্টমার সার্ভিস একদম বাজে, কেউ ফোন ধরে না
🏷️  [('Customer_Service', 0.982)]

📝 নেটওয়ার্ক সমস্যার কারণে পেমেন্ট হয়নি
🏷️  [('Billing', 0.617), ('Network', 0.891)]

📝 লগইন করতে পারতেছি না অ্যাকাউন্টে ঢুকতে পারছি না
🏷️  [('Account', 0.971)]
